In [ ]:
%pip install -q langchain langchain-google-genai python-dotenv requests


# Connect the Gemini API


In [ ]:
import os

# Load the API key from a local .env file (git-ignored) or the shell environment, so this
# notebook runs from any IDE or terminal. In Google Colab, use Secrets instead:
#   from google.colab import userdata
#   GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
except ImportError:
    pass

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "YOUR_API_KEY_HERE")
print("API key loaded:", "yes" if GOOGLE_API_KEY != "YOUR_API_KEY_HERE" else "NO — set it in .env")


# Build a basic langchain agent


In [ ]:
import requests
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# Wikipedia tool. The old `wikipedia` PyPI package sends a shared default User-Agent that
# Wikipedia now rate-limits (HTTP 429 -> JSONDecodeError), so we call the API directly with
# a descriptive User-Agent, which Wikipedia's policy requires.
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = {"User-Agent": "MSAAI-501-Lab/1.0 (gunaphysics@gmail.com)"}

@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return a short plain-text summary of the most relevant article."""
    search = requests.get(WIKI_API, headers=WIKI_HEADERS, timeout=15, params={
        "action": "query", "list": "search", "srsearch": query, "srlimit": 1, "format": "json",
    }).json()
    hits = search.get("query", {}).get("search", [])
    if not hits:
        return f"No Wikipedia article found for '{query}'."
    title = hits[0]["title"]
    page = requests.get(WIKI_API, headers=WIKI_HEADERS, timeout=15, params={
        "action": "query", "prop": "extracts", "exintro": True, "explaintext": True,
        "redirects": 1, "titles": title, "format": "json",
    }).json()
    pages = page.get("query", {}).get("pages", {})
    summary = next(iter(pages.values()), {}).get("extract", "")
    return f"{title}: {summary[:1000]}" if summary else f"Found '{title}' but no summary available."

# Initialize the Gemini model (gemini-1.5-* is retired; use a current model)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=GOOGLE_API_KEY)

# Build a ReAct-style tool-calling agent.
# LangChain 1.x removed initialize_agent; create_agent (built on LangGraph) is the replacement.
agent = create_agent(llm, tools=[search_wikipedia])


# Run the agent


In [ ]:
# Run the agent with a sample query
result = agent.invoke({"messages": [{"role": "user", "content": "What is the capital of France?"}]})

# Gemini returns the answer as content blocks; normalize the final message to plain text.
final = result["messages"][-1]
answer = final.content
if isinstance(answer, list):
    answer = "".join(b.get("text", "") for b in answer if isinstance(b, dict))
print(answer)
